In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time
import serial
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Instruments
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Instruments.VICI_valves.dim import DIM
from Instruments.VICI_valves.fsm import FSM
from Instruments.Syr_pumps.HB_syr_pump import HBElite
from Instruments.Knauer.knauer_pump_azura import KnauerPumpAzura

# Ecosystems
from Ecosystems.reaction_segment_generation import RSG
from Ecosystems.segmentation import Segmentation

# Logger
from Core.logging import flow_logger as logger

# Experiment framework
from Core.experiment_context import ExperimentManager
from Core.experiment_builder import ExperimentBuilder
from Core.experiment_validator import ExperimentValidator
from Core.experiment_compiler import ExperimentCompiler
from Core.experiment_intent import ExperimentIntent
from Core.inventory import Inventory

In [3]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

# --- Gilson ---
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("VB-1-13")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=155.5,
    y_offset=10,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

tray.add_module(
    slot=3,
    name="dim",
    module=dim,
    x_offset=9,     #These values are to be changed
    y_offset=104,
    x_min=0,
    x_max=25,
    y_min=75,
    y_max=120
) 

# --- Ecosystems ---
rsg = RSG(gilson=g, pump=syr_pump, dim=dim)

seg = Segmentation(
    dim=dim,
    fsm=fsm,
    carrier_pump=k_pump,
    rsg=rsg
)

print("Hardware connected.")

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4
[Pump] Flow stopped
[FSM] Valve moved to B -> state = CARRIER_TO_DIM
[DIM] Valve moved to A -> state = DIMState.INJECT
Hardware connected.


In [6]:
k_pump.set_flow_rate(1000)
k_pump.start_flow()

[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started


'ON:OK'

In [7]:
k_pump.stop_flow()

[Pump] Flow stopped


'OFF:OK'

In [3]:
inventory = Inventory()

# Clear + reassign (optional but safer for now)
inventory.assign(
    module="rack1",
    vial=1,
    name="BnOH_stock",
    concentration_M=0.0811,   # adjust if known
    solvent="MeCN",
    current_volume_uL=1500,
    min_safe_volume_uL=200    # These values are approximate at this point
)

inventory.assign(
    module="rack2",
    vial=1,
    name="MeCN",
    concentration_M=None,
    solvent="MeCN",
    current_volume_uL=80000,
    min_safe_volume_uL=20000
)

In [4]:
intent = ExperimentIntent(
    experiment_id="VB-1-13",
    description="10x repeated 100 uL BnOH_stock slug in MeCN"
)

intent.add_block(
    name="BnOH single slug repeats",
    components=["BnOH_stock"],
    ratios=[[100] for _ in range(10)],  # 10 identical slugs
    total_volume_uL=100.0
)

intent.summary()

{'experiment_id': 'VB-1-13',
 'description': '10x repeated 100 uL BnOH_stock slug in MeCN',
 'num_blocks': 1,
 'estimated_slugs': 10,
 'blocks': [{'name': 'BnOH single slug repeats',
   'components': ['BnOH_stock'],
   'num_slugs': 10,
   'total_volume_uL': 100.0,
   'fixed_total_volume': True}]}

In [5]:
compiler = ExperimentCompiler(inventory)

compiled_df = compiler.compile(intent)

compiled_df.head()

,slug_id,module,vial,volume_uL,component,block_id,slug_order,component_order
0,block_1_slug_1,rack1,1,100.0,BnOH_stock,block_1,1,1
1,block_1_slug_2,rack1,1,100.0,BnOH_stock,block_1,2,1
2,block_1_slug_3,rack1,1,100.0,BnOH_stock,block_1,3,1
3,block_1_slug_4,rack1,1,100.0,BnOH_stock,block_1,4,1
4,block_1_slug_5,rack1,1,100.0,BnOH_stock,block_1,5,1


In [6]:
compiled_df[["slug_id", "volume_uL", "component"]]

,slug_id,volume_uL,component
0,block_1_slug_1,100.0,BnOH_stock
1,block_1_slug_2,100.0,BnOH_stock
2,block_1_slug_3,100.0,BnOH_stock
3,block_1_slug_4,100.0,BnOH_stock
4,block_1_slug_5,100.0,BnOH_stock
5,block_1_slug_6,100.0,BnOH_stock
6,block_1_slug_7,100.0,BnOH_stock
7,block_1_slug_8,100.0,BnOH_stock
8,block_1_slug_9,100.0,BnOH_stock
9,block_1_slug_10,100.0,BnOH_stock


In [7]:
builder = ExperimentBuilder(inventory=inventory)

result = builder.build_and_create(
    experiment_id="VB-1-13",
    rows=compiled_df,
    description=intent.description,
    global_conditions={
        "flowrate_ul_min": 1000,
        "gas_prime_s": 20,
        "air_gap_between_uL": 0.0
    },
    overwrite=True
)

print(result["plan_path"])
pd.DataFrame(result["summary"])

C:\Users\CHAD-HPLC\Documents\VictorFlow\Experiments\VB-1-13\plan.json


,slug_id,num_components,total_volume_uL,components
0,block_1_slug_1,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
1,block_1_slug_2,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
2,block_1_slug_3,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
3,block_1_slug_4,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
4,block_1_slug_5,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
5,block_1_slug_6,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
6,block_1_slug_7,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
7,block_1_slug_8,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
8,block_1_slug_9,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"
9,block_1_slug_10,1,100.0,"[(BnOH_stock, rack1, 1, 100.0)]"


In [8]:
manager = ExperimentManager()

manager.load_experiment("VB-1-13")
manager.status()

[EXPERIMENT] VB-1-13 loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] State = prepared
[EXPERIMENT] Awaiting arm_experiment()
Mode: experiment
Experiment: VB-1-13
State: prepared
System state: UNKNOWN
Reactor state: UNKNOWN
RSG state: UNKNOWN
Platform state: UNKNOWN
Slug index: 0


In [9]:
trace = manager.preview_execution(seg, wash_policy="needle")

for step in trace:
    print(step)


--- SLUG 1: block_1_slug_1 ---
seg.prime_gas_path(20.0s)
seg.create_slug(...)
  → RSG.build_rxn_segment()
    → pickup 100.0 uL from rack1 vial 1
    → post-pickup air gap (5.0 uL)
  → dispense 105.0 uL to DIM (100.0 liquid + 0.0 between + 5.0 post-air)
  → switch DIM to inject
  → start carrier flow (1000.0 uL/min)
  → RSG.needle_wash()

--- SLUG 2: block_1_slug_2 ---
seg.reset_for_next_slug()
seg.create_slug(...)
  → RSG.build_rxn_segment()
    → pickup 100.0 uL from rack1 vial 1
    → post-pickup air gap (5.0 uL)
  → dispense 105.0 uL to DIM (100.0 liquid + 0.0 between + 5.0 post-air)
  → switch DIM to inject
  → start carrier flow (1000.0 uL/min)
  → RSG.needle_wash()

--- SLUG 3: block_1_slug_3 ---
seg.reset_for_next_slug()
seg.create_slug(...)
  → RSG.build_rxn_segment()
    → pickup 100.0 uL from rack1 vial 1
    → post-pickup air gap (5.0 uL)
  → dispense 105.0 uL to DIM (100.0 liquid + 0.0 between + 5.0 post-air)
  → switch DIM to inject
  → start carrier flow (1000.0 uL/min)

In [10]:
manager.precondition_reactor(seg, carrier_flowrate_ul_min=1000, stabilisation_time_s=60)

manager.prepare_rsg(
    seg,
    air_gap=20.0,
    rate_wdr=0.10
)

manager.arm_experiment()

[REACTOR] Setting valve positions
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started
[REACTOR] Stabilising...
[SYSTEM] Reactor READY
[RSG] Initialising
[Initialise] Setting up start condition
[Air Gap] 20.0uL @ 0.1mL/min
[Initialise] Ready - air gap in place
[SYSTEM] RSG READY
[EXPERIMENT] VB-1-13 armed
[EXPERIMENT] Awaiting execute_experiment()


In [11]:
preview = manager.preview()
pd.DataFrame(preview)


[EXPERIMENT PREVIEW]
------------------------------------------------------------
1. block_1_slug_1 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
2. block_1_slug_2 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
3. block_1_slug_3 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
4. block_1_slug_4 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
5. block_1_slug_5 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
6. block_1_slug_6 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
7. block_1_slug_7 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
8. block_1_slug_8 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
9. block_1_slug_9 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
10. block_1_slug_10 | 1 components | 100.0 uL
   rack1:1 (100.0 uL)
------------------------------------------------------------
[EXPERIMENT] 10 slugs total



,slug_number,slug_id,num_components,total_volume_uL,components
0,1,block_1_slug_1,1,100.0,rack1:1 (100.0 uL)
1,2,block_1_slug_2,1,100.0,rack1:1 (100.0 uL)
2,3,block_1_slug_3,1,100.0,rack1:1 (100.0 uL)
3,4,block_1_slug_4,1,100.0,rack1:1 (100.0 uL)
4,5,block_1_slug_5,1,100.0,rack1:1 (100.0 uL)
5,6,block_1_slug_6,1,100.0,rack1:1 (100.0 uL)
6,7,block_1_slug_7,1,100.0,rack1:1 (100.0 uL)
7,8,block_1_slug_8,1,100.0,rack1:1 (100.0 uL)
8,9,block_1_slug_9,1,100.0,rack1:1 (100.0 uL)
9,10,block_1_slug_10,1,100.0,rack1:1 (100.0 uL)


In [ ]:
results = manager.execute_experiment(seg, wash_policy="needle_only")
results

[EXPERIMENT] Executing VB-1-13
[FSM] Valve moved to A -> state = GAS_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[Segmentation] Phase: READY -> GAS_PRIMED
[Segmentation] Phase: GAS_PRIMED -> LOOP_LOADING
[DIM] Valve moved to B -> state = DIMState.LOAD
[Build Reaction Segment] block_1_slug_1
[Pickup] 100.0uL from rack1 vial 1 @ 0.05mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127
[Air Gap] 5.0uL @ 0.05mL/min
[DIM] 105.0uL dispensed in DIM @ 0.5mL/min
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0
[Segmentation] Phase: LOOP_LOADING -> LOOP_LOADED
[DIM] Valve moved to A -> state = DIMState.INJECT
[FSM] Valve moved to B -> state = CARRIER_TO_DIM
[k_pump] Flow rate set to 1000.0 uL/min
[Pump] Flow started
[Segmentation] Phase: LOOP_LOADED -> RUNNING
[Segmentation] Segment launched at 1000.0 µL/min


In [4]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=127.0 is already >= target_z=127.0


(np.float64(319.0), np.float64(106.0), 20)

In [5]:
seg.abort()

[Segmentation] ABORT triggered from READY
[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[Segmentation] Phase: READY -> ABORTED
